<a href="https://colab.research.google.com/github/jungmin2197-art/Catholic-lunchmap/blob/main/JMCNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2D 교차 상관(Cross-Correlation)


In [25]:
# d2l 설치를 위한 pip, setuptools, wheel 업그레이드
!pip install --upgrade pip setuptools wheel
# d2l 라이브러리 설치
!pip install d2l==1.0.3 --no-deps
!pip install pandas matplotlib requests
# 라이브러리 임포트
import torch
from torch import nn
from d2l import torch as d2l

# GPU 사용확인
device=d2l.try_gpu()
print(f'현재 사용중인 장치:{device}')
# 교차상관 짜보기
def corr2d(X,K):
  h,w=K.shape #커널 높이 너비
  # 출력크기 계산 공식:(입력-커널+1)
  Y=torch.zeros((X.shape[0]-h+1,X.shape[1]-w+1))

  for i in range(Y.shape[0]):
    for j in range(Y.shape[1]):
      #입력 X에서 커널크기만큼 슬라이스 K와 곱하고 모두더함
      Y[i,j]=(X[i:i+h,j:j+w]*K).sum()
  return Y # Y 값을 반환하도록 수정
# 테스트 책 데이터
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
print(corr2d(X, K))

현재 사용중인 장치:cpu
tensor([[19., 25.],
        [37., 43.]])


# corr2d 함수를 실제 딥러닝 Layer로


In [9]:
class Conv2D(nn.Module):
  def __init__(self,kernel_size):
    super().__init__()
    self.weight=nn.Parameter(torch.rand(kernel_size))
    self.bias=nn.Parameter(torch.zeros(1))

def forward(self,x):
  return corr2d(x,self.weigth)+ self.bias



# Edge 찾기


In [14]:
# 6x8 크기의 모든 원소가 1인 텐서 생성 (흰색 배경)

X=torch.ones((6,8))
# 2열부터 5열까지(인덱스 2:6)를 0으로 설정 (검은색 세로 바)
X[:,2:6]=0
print("--- 입력 이미지 X ---")
print(X)

# 수평 인접 픽셀의 차이 구하는 커널

K=torch.tensor([[1.0,-1.0]])

print("--- 에지 검출 커널 K ---")
print(K)

#위에서 정의한 corr2d 함수사용
Y= corr2d(X,K)

print("--- 에지 검출 결과 Y ---")
print(Y)

# 결과 해석 가이드:
# 1.0이 나온 곳: 흰색(1)에서 검은색(0)으로 변하는 경계
# -1.0이 나온 곳: 검은색(0)에서 흰색(1)으로 변하는 경계
# 0이 나온 곳: 색상 변화가 없는 평평한 곳

# 이미지 X를 전치(Transpose) 시켜서 가로 줄무늬로 만듦
X_transposed = X.t()

# 다시 연산 수행
Y_transposed = corr2d(X_transposed, K)

print("--- 전치된 이미지에 대한 결과 ---")
print(Y_transposed)
# 결과가 모두 0이 나와야 정상입니다! (세로 에지 커널은 가로 에지를 못 잡음)

--- 입력 이미지 X ---
tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])
--- 에지 검출 커널 K ---
tensor([[ 1., -1.]])
--- 에지 검출 결과 Y ---
tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])
--- 전치된 이미지에 대한 결과 ---
tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])


#  커널 학습과 Receptive Field

In [15]:
# 훈련 데이터
# 파이토치 빌트인 레이어는 (Batch, Channel, Height, Width) 4차원을 기대함
X_4d= X.reshape((1,1,6,8))
Y_4d= Y.reshape((1,1,6,7))

# 2. 모델 정의: 1x2 크기의 커널을 가진 합성곱 레이어 생성 (편향 bias는 생략)
conv2d= nn.LazyConv2d(1,kernel_size=(1,2),bias=False)

# 3. 하이퍼파라미터 및 학습 루프
lr=3e-2  # 학습률
for i in range(10):
  Y_hat=conv2d(X_4d) # forward 예측
  loss=(Y_hat - Y_4d)**2  # 손실계산 (MSE)

  conv2d.zero_grad() # 기울기 초기화
  loss.sum().backward() # 역전파 (Backward)

  #가중치 업데이트( 직접 경사하강법 구현)
  conv2d.weight.data[:] -= lr*conv2d.weight.grad

  if(i+1) % 2==0:
    print(f'epoch{i+1},loss{loss.sum():.3f}')
    # 4. 결과 확인: 학습된 커널이 [1.0, -1.0]에 가까운지 확인
print("\n학습된 커널 값:\n", conv2d.weight.data.reshape((1, 2)))

epoch2,loss18.433
epoch4,loss6.535
epoch6,loss2.507
epoch8,loss0.998
epoch10,loss0.404

학습된 커널 값:
 tensor([[ 1.0547, -0.9240]])


# Padding & Stride


In [17]:
#패딩 필요한 이유: 합성곱 반복할수록 이미지 크기가 점점 줄어든다-> 층이깊으면 정보 소멸 문제
# 상하좌우에 특정값 (보통0)을 채워 입력의 크기를 임의로 키운다.

def comp_conv2d(conv2d, X):
  # 입력 X의 차원을 (Batch, Channel, H, W)로 조정 (1, 1, 8, 8)
  X=X.reshape((1,1)+X.shape)
  Y=conv2d(X)
  #연산 후 다시(H,W) 형태로 복구
  return Y.reshape(Y.shape[2:])
# 1. 커널 3x3, 패딩 1 적용 (상하좌우 각 1줄 추가 -> 총 2줄 추가)
# 8 - 3 + 2(패딩) + 1 = 8
conv2d= nn.Conv2d(1,1,kernel_size=3,padding=1)
X=torch.rand(size=(8,8))

output_shape=comp_conv2d(conv2d,X).shape
print(f"입력크기:(8,8) -> 패딩(1) 적용 후 출력 크기:{output_shape}")

# 2. 커널 5x3, 패딩 (2, 1) 적용 높이 너비 다를때
# 높이: 8 - 5 + 2*2 + 1 = 8
# 너비: 8 - 3 + 2*1 + 1 = 8
conv2d_diff = nn.Conv2d(1, 1, kernel_size=(5, 3), padding=(2, 1))
output_shape_diff = comp_conv2d(conv2d_diff, X).shape
print(f"5x3 커널 & (2, 1) 패딩 적용 결과: {output_shape_diff}")

#Stride: 연산 속도를 높이고 데이터를 압축하는 목적
# 책의 헬퍼 함수 정의
def comp_conv2d(conv2d, X):
    # (1, 1)은 배치 크기와 채널 수가 각각 1임을 의미함
    X = X.reshape((1, 1) + X.shape)
    Y = conv2d(X)
    # 첫 두 차원(배치, 채널)을 제거하여 결과 출력
    return Y.reshape(Y.shape[2:])

# 커널 크기 3, 패딩 1, 스트라이드 2 설정
# 입력(8x8) -> 출력(4x4)으로 절반 감소
conv2d = nn.LazyConv2d(1, kernel_size=3, padding=1, stride=2)
X = torch.rand(size=(8, 8))

print(comp_conv2d(conv2d, X).shape)
# 결과: torch.Size([4, 4])

입력크기:(8,8) -> 패딩(1) 적용 후 출력 크기:torch.Size([8, 8])
5x3 커널 & (2, 1) 패딩 적용 결과: torch.Size([8, 8])
torch.Size([4, 4])


# 다중입력 채널


In [27]:
# 입력 데이터 채널수= 커널의 채널수
# 연산 과정: 1. 각 채널별로 대응하는 커널 슬라이스(2D)와 교차 상관 연산을 수행합니다.
# 2. 모든 채널에서 나온 결과(2D 행렬들)를 모두 더합니다

#다중 채널 입력용 교차 상관 함수
def corr2d_multi_in(X,K):
  # X와 K의 0번 차원(채널)을 순회하며 zip으로 묶음
    # 각 채널별 corr2d 결과를 리스트로 만들고 sum으로 모두 더함
    return sum(corr2d(x,k) for x,k in zip(X,K))

    # 입력 X: 2개 채널, 각 채널은 3x3 크기
X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
                  [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])

# 커널 K: 2개 채널, 각 채널은 2x2 크기
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]],
                  [[1.0, 2.0], [3.0, 4.0]]])

# 연산 실행
result = corr2d_multi_in(X, K)
print(result)

# 이전 섹션에서 만든 다중 입력 함수가 필요
def corr2d_multi_in(X, K):
    return sum(corr2d(x, k) for x, k in zip(X, K))

def corr2d(X, K):
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

# 다중 출력 채널 함수
def corr2d_multi_in_out(X, K):
    # K의 0번 차원(출력 채널 수)을 순회하며 각각의 결과물을 쌓음(stack)
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)

# 데이터 준비
X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
                  [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])

# K를 변형하여 3개의 출력 채널용 커널 생성 (3, 2, 2, 2)
K_stack = torch.stack((K, K + 1, K + 2), 0)

print("출력 채널 결과:")
print(corr2d_multi_in_out(X, K_stack))

# 1 * 1 합성곱층: 인접한 픽셀간 관계를 보지않는대신 채널간의 정보를 섞는 역할
# 각 위치 마다 실행되는 fully connect layer와 같다.

def corr2d_multi_in_out_1x1(X,K):
  c_i,h,w=X.shape
  c_o=K.shape[0]
  #데이터를 행렬 곱셈이 가능하도록 펼침
  X=X.reshape((c_i,h * w))
  K=K.reshape((c_o, c_i))
  # 전결합 층의 행렬 곱셈 수행
  Y = torch.matmul(K, X)
  # 다시 원래의 공간 구조로 복원
  return Y.reshape((c_o, h, w))

# 검증
X_rand = torch.normal(0, 1, (3, 3, 3))
K_rand = torch.normal(0, 1, (2, 3, 1, 1))

Y1 = corr2d_multi_in_out_1x1(X_rand, K_rand)
Y2 = corr2d_multi_in_out(X_rand, K_rand)

# 두 방식의 결과가 같은지 확인 (부동소수점 오차 감안)
assert float(torch.abs(Y1 - Y2).sum()) < 1e-6
print("1x1 합성곱 검증 완료!")

tensor([[ 56.,  72.],
        [104., 120.]])
출력 채널 결과:
tensor([[[ 56.,  72.],
         [104., 120.]],

        [[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]]])
1x1 합성곱 검증 완료!


#  Pooling


In [30]:
# Pooling 이동불변성: 물체가 아주조금 움직여도 결과는 크게 변하지않는다
# 다운샘플링: 해상도를 줄여 계산량아끼고, 뒤쪽이 더 넓은걸 보게한다
# 종류: 1 최대 2 평균 최대: 가장 큰값만 남긴다 강한특징 (주로 사용) 평균:전반적인 정보 응ㅇ축

def pool2d(X,pool_size,mode='max'):
  p_h,p_w=pool_size
  # 출력 크기 계산( 입력 - 윈도우 +1)
  Y=torch.zeros((X.shape[0]-p_h+1,X.shape[1]-p_w+1))
  for i in range(Y.shape[0]):
    for j in range(Y.shape[1]):
      if mode=='max':
        Y[i,j]=X[i:i+p_h,j:j+p_w].max()
      elif mode == 'avg':
                Y[i, j] = X[i: i + p_h, j: j + p_w].mean()
    return Y

# 데이터 검증 (Fig. 7.5.1 예제)
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
print("Max Pooling 결과:\n", pool2d(X, (2, 2)))
print("Average Pooling 결과:\n", pool2d(X, (2, 2), 'avg'))

#실사용 할때는 nn.MaxPool2d 사용 프레임워크의 기본설정 확인이 중요하다
# 4x4 입력 생성 (Batch=1, Channel=1)
X = torch.arange(16, dtype=torch.float32).reshape((1, 1, 4, 4))

# 1. 기본 설정: 스트라이드는 풀링 윈도우 크기와 같게 설정됨 (3x3 윈도우 -> 스트라이드 3)
pool2d_default = nn.MaxPool2d(3)
print("Default MaxPool(3x3):\n", pool2d_default(X))

# 2. 패딩과 스트라이드 수동 설정
# (4 + 2*1 - 3) / 2 + 1 = 2.5 -> 내림해서 2
pool2d_custom = nn.MaxPool2d(3, padding=1, stride=2)
print("Custom MaxPool:\n", pool2d_custom(X))

# 3. 비대칭 설정
pool2d_rect = nn.MaxPool2d((2, 3), stride=(2, 3), padding=(0, 1))
print("Rectangular MaxPool:\n", pool2d_rect(X))

Max Pooling 결과:
 tensor([[4., 5.],
        [0., 0.]])
Average Pooling 결과:
 tensor([[2., 3.],
        [0., 0.]])
Default MaxPool(3x3):
 tensor([[[[10.]]]])
Custom MaxPool:
 tensor([[[[ 5.,  7.],
          [13., 15.]]]])
Rectangular MaxPool:
 tensor([[[[ 5.,  7.],
          [13., 15.]]]])


# Lenet 구현

In [33]:

# 1.가중치 초기화 함수(Xavier 초기화)
def init_cnn(module):
  if type(module)==nn.Linear or type(module)==nn.Conv2d:
    nn.init.xavier_uniform_(module.weight)
    # 2. LeNet-5 모델 정의
class LeNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            # 1st Conv: 5x5 커널, 패딩 2 (28x28 유지), 출력 채널 6
            nn.LazyConv2d(6, kernel_size=5, padding=2), nn.Sigmoid(),
            # 1st Pool: 2x2 평균 풀링, 스트라이드 2 (14x14로 감소)
            nn.AvgPool2d(kernel_size=2, stride=2),

            # 2nd Conv: 5x5 커널, 패딩 0 (14-5+1=10x10으로 감소), 출력 채널 16
            nn.LazyConv2d(16, kernel_size=5), nn.Sigmoid(),
            # 2nd Pool: 2x2 평균 풀링, 스트라이드 2 (5x5로 감소)
            nn.AvgPool2d(kernel_size=2, stride=2),

            # Classifier로 넘어가기 위해 3D 텐서를 1D 벡터로 펼침 (16*5*5 = 400)
            nn.Flatten(),

            # Fully Connected Layers
            nn.LazyLinear(120), nn.Sigmoid(),
            nn.LazyLinear(84), nn.Sigmoid(),
            nn.LazyLinear(num_classes)
        )
        # 모든 레이어에 초기화 적용
        self.net.apply(init_cnn)
def forward(self, X):
  return self.net(X)

# 3. 레이어별 출력 크기 확인 함수
def layer_summary(model, X_shape):
    X = torch.randn(*X_shape)
    print(f"입력 크기: {X_shape}")
    print("-" * 30)
    for layer in model.net:
        X = layer(X)
        print(f"{layer.__class__.__name__:<15} 출력 크기: {X.shape}")

# 실행
model = LeNet()
layer_summary(model, (1, 1, 28, 28))



입력 크기: (1, 1, 28, 28)
------------------------------
Conv2d          출력 크기: torch.Size([1, 6, 28, 28])
Sigmoid         출력 크기: torch.Size([1, 6, 28, 28])
AvgPool2d       출력 크기: torch.Size([1, 6, 14, 14])
Conv2d          출력 크기: torch.Size([1, 16, 10, 10])
Sigmoid         출력 크기: torch.Size([1, 16, 10, 10])
AvgPool2d       출력 크기: torch.Size([1, 16, 5, 5])
Flatten         출력 크기: torch.Size([1, 400])
Linear          출력 크기: torch.Size([1, 120])
Sigmoid         출력 크기: torch.Size([1, 120])
Linear          출력 크기: torch.Size([1, 84])
Sigmoid         출력 크기: torch.Size([1, 84])
Linear          출력 크기: torch.Size([1, 10])
